<a href="https://colab.research.google.com/github/GustavoTriaquim/Estrutura-de-dados-nao-lineares/blob/main/AULA04/Aula04_E03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install osmnx folium -q

import osmnx as ox
import networkx as nx
import folium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 2.4 MB/s eta 0:00:00


In [ ]:
def desenhar_pois(gdf, mapa, cor="green"):
  for ix, linha in gdf.iterrows():
    geom = linha.geometry

    if geom.geom_type == "Point":
      lat, lon = geom.y, geom.x
    elif geom.geom_type in ["Polygon", "MultiPolygon"]:
      lat, lon = geom.centroid.y, geom.centroid.x
    else:
      continue

    folium.CircleMarker(
        location=[lat, lon],
        radius=3,
        color=cor,
        fill=True,
        fill_color=cor
    ).add_to(mapa)

# INDÚSTRIAS EM CAMPO LARGO
industria = ox.features_from_place("Campo Largo, Paraná, Brasil", tags={"landuse": "industrial"})

mapa_industria = folium.Map(
    location=ox.geocode("Campo Largo, Paraná, Brasil"),
    zoom_start=12
)

desenhar_pois(industria, mapa_industria, "gray")
mapa_industria

In [ ]:
def desenhar_pois(gdf, mapa, cor="green"):
  for ix, linha in gdf.iterrows():
    geom = linha.geometry

    if geom.geom_type == "Point":
      lat, lon = geom.y, geom.x
    elif geom.geom_type in ["Polygon", "MultiPolygon"]:
      lat, lon = geom.centroid.y, geom.centroid.x
    else:
      continue

    folium.CircleMarker(
        location=[lat, lon],
        radius=3,
        color=cor,
        fill=True,
        fill_color=cor
    ).add_to(mapa)

# ESCOLAS EM SÃO JOSÉ DOS PINHAIS
escolas = ox.features_from_place("São José dos Pinhais, Paraná, Brasil", tags={"amenity": "school"})

mapa_escolas = folium.Map(
    location=ox.geocode("São José dos Pinhais, Paraná, Brasil"),
    zoom_start=12
)

desenhar_pois(escolas, mapa_escolas, "blue")
mapa_escolas

In [ ]:
def desenhar_pois(gdf, mapa, cor="green"):
  for ix, linha in gdf.iterrows():
    geom = linha.geometry

    if geom.geom_type == "Point":
      lat, lon = geom.y, geom.x
    elif geom.geom_type in ["Polygon", "MultiPolygon"]:
      lat, lon = geom.centroid.y, geom.centroid.x
    else:
      continue

    folium.CircleMarker(
        location=[lat, lon],
        radius=3,
        color=cor,
        fill=True,
        fill_color=cor
    ).add_to(mapa)

# POSTOS DE SAÚDE NA FAZENDA RIO GRANDE
saude = ox.features_from_place("Fazenda Rio Grande, Paraná, Brasil", tags={"amenity": ["clinic", "hospital"]})

mapa_saude = folium.Map(
    location=ox.geocode("Fazenda Rio Grande, Paraná, Brasil"),
    zoom_start=12
)

desenhar_pois(saude, mapa_saude, "red")
mapa_saude

In [ ]:
bairro = "São José dos Pinhais, Paraná, Brasil"

G_sjp = ox.graph_from_place(
    bairro,
    network_type="drive",
    truncate_by_edge=True,
    retain_all=True,
    simplify=False
)

postos_policiais = ox.features_from_place("São José dos Pinhais, Paraná, Brasil", tags={"amenity": "police"})

unisenai_coord = ox.geocode("Unisenai, São José dos Pinhais, PR")
no_unisenai = ox.distance.nearest_nodes(G_sjp, X=unisenai_coord[1], Y=unisenai_coord[0])

postos_coords = []
for idx, linha in postos_policiais.iterrows():
  geom = linha.geometry

  if geom.geom_type == "Point":
    postos_coords.append((linha.get("name", "Posto sem nome"), geom.y, geom.x))
  elif geom.geom_type in ["Polygon", "MultiPolygon"]:
    postos_coords.append((linha.get("name", "Posto sem nome"), geom.centroid.y, geom.centroid.x))

mais_proximo = None
menor_distancia = float("inf")
for nome, lat, lon in postos_coords:
  no_posto = ox.distance.nearest_nodes(G_sjp, X=lon, Y=lat)
  distancia = nx.shortest_path_length(G_sjp, source=no_unisenai, target=no_posto, weight="length")

  if distancia < menor_distancia:
    menor_distancia = distancia
    mais_proximo = (nome, lat, lon, no_posto)

print(f"Posto mais próximo: {mais_proximo[0]} - {menor_distancia:.0f} metros")

rota = nx.shortest_path(G_sjp, source=no_unisenai, target=mais_proximo[3], weight="length")
pontos_rota = [[G_sjp.nodes[n]['y'], G_sjp.nodes[n]['x']] for n in rota]

mapa_policia = folium.Map(
    location=unisenai_coord,
    zoom_start=13
)
for nome, lat, lon in postos_coords:
  folium.CircleMarker(
      location=[lat, lon],
      radius=4,
      color="blue",
      fill=True,
      popup=nome
  ).add_to(mapa_policia)

folium.Marker(
    unisenai_coord,
    popup="Unisenai",
    icon=folium.Icon(colo="green")
).add_to(mapa_policia)

folium.PolyLine(
    locations=pontos_rota,
    color="red",
    weight=4,
    opacity=0.85,
    tooltip="Rota até o posto mais próximo"
).add_to(mapa_policia)

mapa_policia

Posto mais próximo: 1ª Delegacia Regional de Polícia de São José dos Pinhais - 3323 metros
